# Machine Learning for Anti-Money Laundering

***
**Master's thesis: empirical part**

**Notebook 2: Feature Construction and Feature Selection**

Benchmark: IBM AMLworld, variants HI-Small and LI-Small (Altman et al., 2023)
***

| Notebook | Content | Chapter 4 |
| --- | --- | --- |
| 01 Data and EDA | acquisition, integrity checks, temporal partition, EDA | 4.2–4.5 |
| **02 Features and feature selection** (this notebook) | Sets A, B, C; causality verification; screening | 4.6 |
| 03 Models | one section per family: search, fit, threshold, evaluation | 4.7–4.8 |
| 04 Results | comparison across families, sets and variants | 4.9 |

## Preliminaries

This notebook builds the three nested feature sets of Section 4.6 for both variants, verifies
that no feature uses information from the moment of its own transaction or later, screens the
features on the training block, and stores one feature matrix per variant on Drive for
Notebook 03.

**Requirements:** Notebook 01 must have run in the same Google account, because this notebook
reads its prepared data (`MyDrive/aml-thesis/data/interim/`) and its code (`MyDrive/aml-thesis/aml/`).
Select a CPU runtime with the default memory and use *Runtime → Run all*. Building the graph
features takes several minutes per variant.

In [ ]:
# Parameters
VARIANTS    = ["HI-Small", "LI-Small"]   # feature matrices are built for both
EDA_VARIANT = "HI-Small"                 # the variant whose features are explored below
SEED        = 0                          # used only for plotting and diagnostic samples
DRY_RUN     = False                      # True: synthetic data in a separate folder

In [ ]:
# Import relevant Python packages
import os, sys, json, gc, platform, subprocess, datetime
from pathlib import Path

import numpy as np
import pandas as pd
from pandas import set_option
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

set_option("display.precision", 3)
set_option("display.width", 120)
set_option("display.max_columns", 30)
sns.set_theme(context="notebook", style="whitegrid", palette="viridis")

In [ ]:
# Google Drive, project folder and a record of the computational environment
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive")
except ImportError:                      # outside Colab: work next to the notebook
    BASE = Path.cwd()
PROJECT = BASE / ("aml-thesis-dryrun" if DRY_RUN else "aml-thesis")
assert (PROJECT / "aml" / "prepare.py").exists(), \
    f"{PROJECT} has no project code: run Notebook 01 first (same Google account, same DRY_RUN)"
os.chdir(PROJECT)
for d in ["data/features", "results/figures", "results/tables"]:
    (PROJECT / d).mkdir(parents=True, exist_ok=True)
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / 2**30
except ImportError:
    ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
header = (f"# recorded {datetime.datetime.now().isoformat(timespec='seconds')} (Notebook 02)\n"
          f"# python {platform.python_version()} | cpus {os.cpu_count()} | ram_gb {ram_gb:.1f}\n")
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"],
                        capture_output=True, text=True).stdout
(PROJECT / "results" / "environment_nb02.txt").write_text(header + freeze)
print(f"project: {PROJECT}")
print(header)

### Project code

The cells below add five modules to the `aml` package on Drive; the modules written by
Notebook 01 are used unchanged. As before, pipeline logic lives only in these modules and is
covered by tests, while the exploratory plots further down are written inline.

In [ ]:
%%writefile aml/fx.py
"""Conversion of amounts into one currency (Section 4.6).

AMLworld records each amount in its own currency, so raw amounts are not comparable
across currencies. A cross-currency transaction reveals the exchange rate between its two
currencies (amount received / amount paid). The median rate per currency pair, estimated
on the training block only, links every currency to the base currency through the chain
of observed pairs.
"""
from collections import deque

import numpy as np
import pandas as pd

from .features import _same_value


def implied_rates(train: pd.DataFrame, base: str) -> tuple[pd.Series, pd.DataFrame, pd.DataFrame]:
    """Return base-currency units per unit of each currency, a per-currency diagnostics
    table, and the per-pair median log rates with their interquartile range."""
    currencies = sorted(set(train["Payment Currency"].astype(str).unique())
                        | set(train["Receiving Currency"].astype(str).unique()))
    cross = ~_same_value(train["Payment Currency"], train["Receiving Currency"])
    sub = train.loc[cross, ["Payment Currency", "Receiving Currency",
                            "Amount Paid", "Amount Received"]]
    sub = sub[(sub["Amount Paid"] > 0) & (sub["Amount Received"] > 0)]
    k = np.log(sub["Amount Received"].astype(float) / sub["Amount Paid"].astype(float))
    pairs = (pd.DataFrame({"pay": sub["Payment Currency"].astype(str).to_numpy(),
                           "rec": sub["Receiving Currency"].astype(str).to_numpy(),
                           "k": k.to_numpy()})
             .groupby(["pay", "rec"])["k"]
             .agg(n="size", log_rate="median",
                  iqr=lambda x: float(np.subtract(*np.percentile(x, [75, 25]))))
             .reset_index()
             .sort_values(["n", "pay", "rec"], ascending=[False, True, True]))

    # 1 unit of `pay` = exp(k) units of `rec`  =>  lu(pay) = k + lu(rec)
    log_usd = {base: 0.0} if base in currencies else {}
    if not log_usd and len(currencies) == 1:
        log_usd = {currencies[0]: 0.0}
    queue = deque(log_usd)
    while queue:
        c = queue.popleft()
        for p, r, kk in pairs[["pay", "rec", "log_rate"]].itertuples(index=False):
            if p == c and r not in log_usd:
                log_usd[r] = log_usd[c] - kk
                queue.append(r)
            elif r == c and p not in log_usd:
                log_usd[p] = kk + log_usd[c]
                queue.append(p)

    rates = pd.Series({c: float(np.exp(v)) for c, v in log_usd.items()}, dtype=float)
    method = {c: "implied rate" for c in rates.index}
    missing = [c for c in currencies if c not in rates.index]
    if missing:   # no chain of cross-currency pairs: scale-match medians instead (flagged)
        known = train["Payment Currency"].astype(str).isin(rates.index)
        ref = float(np.median(train.loc[known, "Amount Paid"].astype(float).to_numpy()
                              * train.loc[known, "Payment Currency"].astype(str)
                              .map(rates).to_numpy())) if known.any() else 1.0
        for c in missing:
            med = float(train.loc[train["Payment Currency"].astype(str) == c,
                                  "Amount Paid"].median())
            rates[c] = ref / med if med > 0 else np.nan
            method[c] = "median scaling (no linking pair)"
    diag = pd.DataFrame({"base_units_per_unit": rates, "method": pd.Series(method)})
    diag.index.name = "currency"
    return rates.sort_index(), diag.sort_index(), pairs.reset_index(drop=True)


def to_base(amount: pd.Series, currency: pd.Series, rates: pd.Series) -> np.ndarray:
    """Multiply each amount by the rate of its currency (via categorical codes)."""
    cats = currency.cat.categories.astype(str)
    r = np.append(rates.reindex(cats).to_numpy(dtype=float), np.nan)  # code -1 -> NaN
    return amount.to_numpy(dtype=float) * r[currency.cat.codes.to_numpy()]


In [ ]:
%%writefile aml/aggregates.py
"""Set B: per-account aggregates over a look-back window (Section 4.6).

Every aggregate for a transaction at minute t uses only events at minutes t' with
t - W <= t' < t. Events in the same minute as the transaction are excluded, including
other transactions of that minute, so ties in the minute-resolution timestamps cannot
leak information. All windows are evaluated exactly, without loops over transactions:
events are sorted by (account, minute), window bounds are found by binary search, and
sums come from prefix sums that restart at every account (so large totals of other
accounts cannot erode precision).
"""
import numpy as np
import pandas as pd

K = np.int64(1) << np.int64(32)          # key = account * K + minute
MINUTE = pd.Timedelta("1min")


def minutes(ts: pd.Series, origin: pd.Timestamp) -> np.ndarray:
    return ((ts - origin) // MINUTE).to_numpy(dtype=np.int64)


def window_minutes(window: str) -> int:
    return int(pd.Timedelta(window) / MINUTE)


def account_codes(df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, int]:
    """Codes for source and destination accounts on one shared index."""
    cats = df["src_account"].cat.categories.union(df["dst_account"].cat.categories)
    src = df["src_account"].cat.set_categories(cats).cat.codes.to_numpy(np.int64)
    dst = df["dst_account"].cat.set_categories(cats).cat.codes.to_numpy(np.int64)
    return src, dst, len(cats)


class EventIndex:
    """Events of one kind sorted by (account, minute), with per-account prefix sums."""

    def __init__(self, acct: np.ndarray, t: np.ndarray, values: dict | None = None):
        key = acct * K + t
        order = np.argsort(key, kind="stable")
        self.key, self.t = key[order], t[order]
        acct_sorted = acct[order]
        self.prefix = {name: pd.Series(np.asarray(v, dtype=np.float64)[order])
                       .groupby(acct_sorted, sort=False).cumsum().to_numpy()
                       for name, v in (values or {}).items()}

    def bounds(self, a: np.ndarray, t: np.ndarray, w: int | None):
        start = np.searchsorted(self.key, a * K, "left")
        hi = np.searchsorted(self.key, a * K + t, "left")              # t' < t
        lo = start if w is None else np.maximum(
            np.searchsorted(self.key, a * K + (t - w), "left"), start)  # t' >= t - w
        return start, lo, hi

    def window(self, a, t, w):
        start, lo, hi = self.bounds(a, t, w)

        def wsum(name):
            p = self.prefix[name]
            upper = np.where(hi > start, p[np.maximum(hi - 1, 0)], 0.0)
            lower = np.where(lo > start, p[np.maximum(lo - 1, 0)], 0.0)
            return upper - lower
        return hi - lo, wsum

    def previous_minute(self, a, t):
        start, _, hi = self.bounds(a, t, None)
        return np.where(hi > start, self.t[np.maximum(hi - 1, 0)], -1)


def distinct_counterparties(acct, cp, t, q_acct, q_t, w, n_accounts):
    """Number of distinct counterparties of q_acct with an event in [q_t - w, q_t).

    For one (account, counterparty) pair, the indicator is 1 exactly for query minutes in
    the union of the intervals (e, e + w] over its event minutes e. Overlapping intervals
    are merged per pair; the count for a query is then (#merged intervals starting before
    q_t) - (#merged intervals ending before q_t), both found by binary search.
    """
    pair = acct * np.int64(n_accounts) + cp
    order = np.lexsort((t, pair))
    p, tt, a = pair[order], t[order], acct[order]
    new = np.ones(len(p), dtype=bool)
    new[1:] = (p[1:] != p[:-1]) | (tt[1:] > tt[:-1] + w)
    first = np.flatnonzero(new)
    last = np.append(first[1:] - 1, len(p) - 1)
    S = np.sort(a[first] * K + tt[first])
    F = np.sort(a[first] * K + tt[last] + w)
    base = q_acct * K
    n_start = np.searchsorted(S, base + q_t, "left") - np.searchsorted(S, base, "left")
    n_end = np.searchsorted(F, base + q_t, "left") - np.searchsorted(F, base, "left")
    return n_start - n_end


def _moments(idx: EventIndex, a, t, w):
    cnt, wsum = idx.window(a, t, w)
    total, s1, s2 = wsum("amt"), wsum("lg"), wsum("lg2")
    mean = np.where(cnt > 0, s1 / np.maximum(cnt, 1), 0.0)
    var = np.where(cnt > 1, (s2 - cnt * mean ** 2) / np.maximum(cnt - 1, 1), 0.0)
    return cnt, total, mean, np.sqrt(np.clip(var, 0.0, None))


def set_b(src, dst, t, amt_out, amt_in, w: int, n_accounts: int) -> pd.DataFrame:
    """Fourteen Set B features. Amounts are in the base currency; means and standard
    deviations are taken of log(1 + amount), which is numerically stable and robust to
    amounts that span many orders of magnitude."""
    lg_out, lg_in = np.log1p(amt_out), np.log1p(amt_in)
    out = EventIndex(src, t, {"amt": amt_out, "lg": lg_out, "lg2": lg_out ** 2})
    inc = EventIndex(dst, t, {"amt": amt_in, "lg": lg_in, "lg2": lg_in ** 2})
    every = EventIndex(np.concatenate([src, dst]), np.concatenate([t, t]))

    so_cnt, so_sum, so_mean, so_std = _moments(out, src, t, w)
    di_cnt, di_sum, di_mean, di_std = _moments(inc, dst, t, w)
    si_cnt = inc.window(src, t, w)[0]
    do_cnt = out.window(dst, t, w)[0]

    def hours_since_previous(a):
        prev = every.previous_minute(a, t)
        # no earlier activity: hours since the start of observation (a lower bound)
        return np.where(prev >= 0, t - prev, t) / 60.0

    feats = {
        "src_out_cnt": so_cnt, "src_out_sum": so_sum,
        "src_out_mean_log": so_mean, "src_out_std_log": so_std,
        "src_out_n_cp": distinct_counterparties(src, dst, t, src, t, w, n_accounts),
        "dst_in_cnt": di_cnt, "dst_in_sum": di_sum,
        "dst_in_mean_log": di_mean, "dst_in_std_log": di_std,
        "dst_in_n_cp": distinct_counterparties(dst, src, t, dst, t, w, n_accounts),
        "src_in_share": (si_cnt + 1) / (si_cnt + so_cnt + 2),
        "dst_in_share": (di_cnt + 1) / (di_cnt + do_cnt + 2),
        "src_hours_since_prev": hours_since_previous(src),
        "dst_hours_since_prev": hours_since_previous(dst),
    }
    return pd.DataFrame({k: np.asarray(v, dtype=np.float32) for k, v in feats.items()})


def verify_b(src, dst, t, amt_out, amt_in, B: pd.DataFrame, w: int, rows: np.ndarray) -> dict:
    """Recompute Set B for the given rows by direct filtering of each account's events
    and compare with the vectorised result (replaces Listing 4.4)."""
    by_src = np.argsort(src, kind="stable")
    by_dst = np.argsort(dst, kind="stable")
    src_s, dst_s = src[by_src], dst[by_dst]

    def events(sorted_codes, order, a):
        i, j = np.searchsorted(sorted_codes, a, "left"), np.searchsorted(sorted_codes, a, "right")
        return order[i:j]

    got_all = {c: B[c].to_numpy() for c in B.columns}
    worst = {c: 0.0 for c in B.columns}
    for r in rows:
        a, b, tr = src[r], dst[r], t[r]
        e_out = events(src_s, by_src, a)
        e_out = e_out[(t[e_out] >= tr - w) & (t[e_out] < tr)]
        e_in = events(dst_s, by_dst, b)
        e_in = e_in[(t[e_in] >= tr - w) & (t[e_in] < tr)]
        e_src_in = events(dst_s, by_dst, a)
        e_dst_out = events(src_s, by_src, b)
        n_si = int(((t[e_src_in] >= tr - w) & (t[e_src_in] < tr)).sum())
        n_do = int(((t[e_dst_out] >= tr - w) & (t[e_dst_out] < tr)).sum())

        def mom(v):
            lg = np.log1p(v)
            return (len(v), v.sum(), lg.mean() if len(v) else 0.0,
                    lg.std(ddof=1) if len(v) > 1 else 0.0)

        def prev(a_):
            ts = np.concatenate([t[events(src_s, by_src, a_)], t[events(dst_s, by_dst, a_)]])
            ts = ts[ts < tr]
            return (tr - ts.max()) / 60.0 if len(ts) else tr / 60.0

        c1, s1, m1, d1 = mom(amt_out[e_out])
        c2, s2, m2, d2 = mom(amt_in[e_in])
        ref = {"src_out_cnt": c1, "src_out_sum": s1, "src_out_mean_log": m1,
               "src_out_std_log": d1, "src_out_n_cp": len(np.unique(dst[e_out])),
               "dst_in_cnt": c2, "dst_in_sum": s2, "dst_in_mean_log": m2,
               "dst_in_std_log": d2, "dst_in_n_cp": len(np.unique(src[e_in])),
               "src_in_share": (n_si + 1) / (n_si + c1 + 2),
               "dst_in_share": (c2 + 1) / (c2 + n_do + 2),
               "src_hours_since_prev": prev(a), "dst_hours_since_prev": prev(b)}
        for c, v in ref.items():
            got = float(got_all[c][r])
            scale = max(1.0, abs(v))
            worst[c] = max(worst[c], abs(got - v) / scale)
    # float32 storage: relative error up to ~1e-6 is rounding, not a discrepancy
    return {"n_rows": int(len(rows)), "max_relative_error": worst,
            "passed": bool(max(worst.values()) < 1e-4)}


In [ ]:
%%writefile aml/graph.py
"""Set C: structural statistics of the account graph from daily snapshots (Section 4.6).

Snapshot k is taken at minute s_k (a midnight) and contains the directed edges of all
transactions with s_k - window <= t < s_k; self-transfers are excluded. A transaction at
minute t inherits the most recent snapshot with s_k <= t, so every edge it can see is
strictly earlier than the transaction. Transactions before the first snapshot fall in the
burn-in period and receive the values of an isolated account.

Degree, counterparty count, reciprocity, the two-hop neighbourhood and the clustering
coefficient are computed with SciPy sparse matrices (row-chunked, so hubs cannot exhaust
memory); PageRank and weakly connected components with NetworkX.
"""
import time

import networkx as nx
import numpy as np
import pandas as pd
import scipy.sparse as sp

ROLES = ("src", "dst")
STATS = ("in_deg", "out_deg", "n_cp", "two_hop", "clustering", "reciprocity",
         "pagerank", "component_size")
ISOLATED = {"in_deg": 0.0, "out_deg": 0.0, "n_cp": 0.0, "two_hop": 0.0,
            "clustering": 0.0, "reciprocity": 0.0, "pagerank": 0.0, "component_size": 1.0}


def snapshot_minutes(origin: pd.Timestamp, t_max: int, freq: str) -> np.ndarray:
    """Snapshot times in minutes after `origin`, anchored at midnights."""
    step = int(pd.Timedelta(freq) / pd.Timedelta("1min"))
    first = int((origin.floor("D") + pd.Timedelta("1D") - origin) / pd.Timedelta("1min"))
    return np.arange(first, t_max + 1, step, dtype=np.int64)


def node_stats(u: np.ndarray, v: np.ndarray, chunk_budget: int = 20_000_000) -> tuple:
    """Statistics of every node of the directed graph with edges u -> v.

    Returns (nodes, stats) with stats[name][i] the value for nodes[i]. Repeated edges
    count once; self-loops must already be removed.
    """
    n_all = np.int64(max(u.max(initial=0), v.max(initial=0)) + 1)
    keys = np.unique(u * n_all + v)
    uu, vv = keys // n_all, keys % n_all
    nodes = np.unique(np.concatenate([uu, vv]))
    iu, iv = np.searchsorted(nodes, uu), np.searchsorted(nodes, vv)
    n = len(nodes)

    out_deg = np.bincount(iu, minlength=n).astype(np.float64)
    in_deg = np.bincount(iv, minlength=n).astype(np.float64)
    A = sp.csr_matrix((np.ones(2 * len(iu)), (np.concatenate([iu, iv]),
                                              np.concatenate([iv, iu]))), shape=(n, n))
    A.sum_duplicates()
    A.data[:] = 1.0
    deg = np.diff(A.indptr).astype(np.float64)        # distinct counterparties

    reverse_exists = np.isin(vv * n_all + uu, keys)
    reciprocated = np.bincount(iu[reverse_exists], minlength=n)
    reciprocity = np.divide(reciprocated, deg, out=np.zeros(n), where=deg > 0)

    # two-hop neighbourhood and triangles, in row chunks bounded by the walk count
    walks = A @ deg
    cost = np.cumsum(walks + deg + 1.0)
    cuts = np.searchsorted(cost, np.arange(1, int(cost[-1] // chunk_budget) + 1) * chunk_budget)
    bounds = np.unique(np.concatenate([[0], np.minimum(cuts + 1, n), [n]]))
    two_hop, tri = np.zeros(n), np.zeros(n)
    for start, stop in zip(bounds[:-1], bounds[1:]):
        R = A[start:stop]
        P = R @ A                                       # length-2 walks
        self_hit = (deg[start:stop] > 0).astype(np.float64)   # walk v -> w -> v
        two_hop[start:stop] = np.diff((P + R).indptr) - self_hit
        tri[start:stop] = np.asarray(P.multiply(R).sum(axis=1)).ravel() / 2.0
    clustering = np.divide(2.0 * tri, deg * (deg - 1), out=np.zeros(n), where=deg >= 2)

    G = nx.DiGraph()
    G.add_nodes_from(range(n))
    G.add_edges_from(zip(iu.tolist(), iv.tolist()))
    pr = nx.pagerank(G, alpha=0.85)
    pagerank = np.fromiter((pr[i] for i in range(n)), dtype=np.float64, count=n) * n
    component = np.zeros(n)
    for comp in nx.weakly_connected_components(G):
        idx = np.fromiter(comp, dtype=np.int64)
        component[idx] = len(idx)

    stats = {"in_deg": in_deg, "out_deg": out_deg, "n_cp": deg, "two_hop": two_hop,
             "clustering": clustering, "reciprocity": reciprocity,
             "pagerank": pagerank, "component_size": component}
    return nodes, stats


def set_c(src, dst, t, origin: pd.Timestamp, freq: str, window: int,
          progress=print) -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray]:
    """Sixteen Set C features (eight statistics for each of the two accounts), the
    per-snapshot report, and the snapshot minute assigned to every row (-1: none)."""
    snaps = snapshot_minutes(origin, int(t.max()), freq)
    assign = np.searchsorted(snaps, t, side="right") - 1
    out = {f"{r}_{s}": np.full(len(t), ISOLATED[s], dtype=np.float32)
           for r in ROLES for s in STATS}
    report = []
    for k, s in enumerate(snaps):
        rows = np.flatnonzero(assign == k)
        if len(rows) == 0:
            continue
        t0 = time.perf_counter()
        e = (t >= s - window) & (t < s) & (src != dst)
        if e.any():
            nodes, stats = node_stats(src[e], dst[e])
            for role, codes in zip(ROLES, (src[rows], dst[rows])):
                pos = np.searchsorted(nodes, codes)
                hit = pos < len(nodes)
                hit[hit] = nodes[pos[hit]] == codes[hit]
                for name in STATS:
                    out[f"{role}_{name}"][rows[hit]] = stats[name][pos[hit]]
            n_nodes, n_edges = len(nodes), int(e.sum())
            max_edge = int(t[e].max())
        else:
            n_nodes, n_edges, max_edge = 0, 0, -1
        secs = time.perf_counter() - t0
        report.append({"snapshot_minute": int(s), "n_edges": n_edges, "n_nodes": n_nodes,
                       "n_rows_served": int(len(rows)), "max_edge_minute": max_edge,
                       "edges_strictly_before": bool(max_edge < s), "seconds": round(secs, 1)})
        progress(f"  snapshot {k + 1}/{len(snaps)}: {n_edges:,} edges, {n_nodes:,} nodes, "
                 f"{len(rows):,} rows, {secs:.1f} s")
    snap_of_row = np.where(assign >= 0, snaps[np.maximum(assign, 0)], -1)
    return pd.DataFrame(out), pd.DataFrame(report), snap_of_row


def verify_c(src, dst, t, C: pd.DataFrame, snap_of_row, window: int, rows) -> dict:
    """For the given rows, recompute degree and counterparty counts directly from the raw
    transactions of the snapshot window, and confirm the snapshot precedes the row."""
    bad, checked = [], 0
    cols = {c: C[c].to_numpy() for c in ["src_out_deg", "src_in_deg", "src_n_cp",
                                         "dst_in_deg", "dst_out_deg", "dst_n_cp"]}
    rows = np.asarray(rows)
    for s in np.unique(snap_of_row[rows]):
        if s < 0:
            continue
        e = (t >= s - window) & (t < s) & (src != dst)
        es, ed = src[e], dst[e]
        for r in rows[snap_of_row[rows] == s]:
            checked += 1
            if s > t[r]:
                bad.append(("snapshot after row", int(r)))
                continue
            for role, a in (("src", src[r]), ("dst", dst[r])):
                outs, ins = np.unique(ed[es == a]), np.unique(es[ed == a])
                ref = {"out_deg": len(outs), "in_deg": len(ins),
                       "n_cp": len(np.union1d(outs, ins))}
                for name, val in ref.items():
                    if cols[f"{role}_{name}"][r] != val:
                        bad.append((f"{role}_{name}", int(r)))
    return {"n_rows_checked": checked, "n_mismatches": len(bad),
            "examples": bad[:5], "passed": not bad}


In [ ]:
%%writefile aml/screening.py
"""Feature screening on the training block (Section 4.6, screening rule).

The rule is fixed in advance and applied once, identically for every model family:
  1. constant features are dropped;
  2. features are visited in their declared order (Set A, then B, then C); a feature
     whose |Spearman rho| with an already kept feature exceeds the threshold is dropped.
Because earlier features always win, a Set B or C feature can be dropped for duplicating
a Set A feature, but never the reverse, so the three sets stay nested.
"""
import numpy as np
import pandas as pd


def spearman(X: pd.DataFrame) -> pd.DataFrame:
    R = np.array(X.rank(method="average").to_numpy(), dtype=np.float64)   # writable copy
    R -= R.mean(axis=0)
    sd = R.std(axis=0)
    R /= np.where(sd > 0, sd, 1.0)
    return pd.DataFrame(R.T @ R / len(R), index=X.columns, columns=X.columns)


def screen(X: pd.DataFrame, order: list, threshold: float) -> tuple[list, dict, pd.DataFrame]:
    constant = [c for c in order if X[c].nunique(dropna=False) <= 1]
    candidates = [c for c in order if c not in constant]
    rho = spearman(X[candidates])
    kept, redundant = [], {}
    for c in candidates:
        if kept:
            r = rho.loc[c, kept].abs()
            if r.max() > threshold:
                redundant[c] = {"duplicates": r.idxmax(), "abs_rho": round(float(r.max()), 4)}
                continue
        kept.append(c)
    report = {"threshold": threshold, "n_rows": int(len(X)), "n_candidates": len(order),
              "constant": constant, "redundant": redundant, "n_kept": len(kept)}
    return kept, report, rho


In [ ]:
%%writefile aml/build_features.py
"""Notebook 02 pipeline: Sets A, B and C, causality verification, screening, persistence.

Usage:  python -m aml.build_features --variant HI-Small
"""
import argparse
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

from . import aggregates as agg
from . import config, fx, graph, screening
from . import feature_params as P
from .features import base_attributes

FEATURES_DIR = config.ROOT / "data" / "features"
ONE_HOT = {"payment_currency": "cur_pay", "receiving_currency": "cur_rec",
           "payment_format": "fmt"}
READ_COLS = ["Timestamp", "src_account", "dst_account", "From Bank", "To Bank",
             "Amount Paid", "Amount Received", "Payment Currency", "Receiving Currency",
             "Payment Format", "Is Laundering", "block", "file_row"]
WRITE_CHUNK = 500_000        # rows per Parquet write, to bound memory


def tail_flag(ts: pd.Series, share: float) -> tuple[np.ndarray, pd.Timestamp]:
    """Flag rows after the last calendar day whose transaction count reaches `share` of the
    median daily count. Uses timestamps only, so no label informs the rule."""
    day = ts.dt.floor("D")
    counts = day.value_counts().sort_index()
    last_regular = counts.index[(counts >= share * counts.median()).to_numpy()].max()
    return (day > last_regular).to_numpy(), last_regular


def set_a(df: pd.DataFrame, rates: pd.Series, train: np.ndarray,
          day_of_week: str = "weekend") -> pd.DataFrame:
    """Set A, encoded: amounts in the base currency, binary flags, hour, the day-of-week
    variant chosen in feature_params, and one indicator per category observed in the
    training block (unseen categories in later blocks are all-zero, so no information
    from them reaches the encoding)."""
    raw = base_attributes(df)
    f32 = lambda a: np.asarray(a, dtype=np.float32)
    out = {"amount_paid_base": f32(fx.to_base(df["Amount Paid"], df["Payment Currency"], rates)),
           "amount_received_base": f32(fx.to_base(df["Amount Received"],
                                                  df["Receiving Currency"], rates)),
           "currency_mismatch": f32(raw["currency_mismatch"]),
           "cross_bank": f32(raw["cross_bank"]),
           "hour": f32(raw["hour"])}
    if day_of_week == "weekend":
        out["weekend"] = f32(raw["day_of_week"].to_numpy() >= 5)
    elif day_of_week == "full":
        out["day_of_week"] = f32(raw["day_of_week"])
    elif day_of_week != "drop":
        raise ValueError(f"DAY_OF_WEEK must be 'weekend', 'drop' or 'full', not {day_of_week!r}")
    for col, prefix in ONE_HOT.items():
        s = raw[col]
        codes = s.cat.codes.to_numpy()
        for c in s[train].cat.remove_unused_categories().cat.categories:
            out[f"{prefix}={str(c).strip().replace(' ', '_')}"] = \
                f32(codes == s.cat.categories.get_loc(c))
    del raw
    return pd.DataFrame(out)


def build(variant: str, interim_dir=config.DATA_INTERIM, features_dir=FEATURES_DIR,
          results_dir=config.RESULTS, progress=print) -> dict:
    t_start = time.perf_counter()
    interim_dir, features_dir, results_dir = map(Path, (interim_dir, features_dir, results_dir))
    tables = results_dir / "tables"
    for d in (features_dir, tables):
        d.mkdir(parents=True, exist_ok=True)
    timing = {}

    df = pd.read_parquet(interim_dir / f"{variant}.parquet", columns=READ_COLS)
    origin = df["Timestamp"].min()
    t = agg.minutes(df["Timestamp"], origin)
    train = (df["block"] == "train").to_numpy()
    burn_in = t < agg.window_minutes(P.BURN_IN)
    tail, last_regular_day = tail_flag(df["Timestamp"], P.TAIL_SHARE)
    rng = np.random.default_rng(P.SEED)

    tic = time.perf_counter()
    rates, fx_diag, fx_pairs = fx.implied_rates(df.loc[train], P.BASE_CURRENCY)
    fx_diag.to_csv(tables / f"{variant}_fx_rates.csv")
    fx_pairs.to_csv(tables / f"{variant}_fx_pairs.csv", index=False)
    A = set_a(df, rates, train, P.DAY_OF_WEEK)
    timing["set_a"] = round(time.perf_counter() - tic, 1)
    progress(f"Set A: {A.shape[1]} columns ({timing['set_a']} s)")

    tic = time.perf_counter()
    src, dst, n_acc = agg.account_codes(df)
    n = len(df)
    file_row = df["file_row"].to_numpy()
    block = pd.Categorical(df["block"])
    label = df["Is Laundering"].to_numpy(np.int8)
    tail_info = {"last_regular_day": str(last_regular_day.date()), "n_rows": int(tail.sum()),
                 "n_laundering": int(label[tail].sum()),
                 "blocks": pd.Series(block[tail]).astype(str).value_counts().to_dict()}
    del df                                            # only arrays are needed from here on
    gc.collect()
    w = agg.window_minutes(P.WINDOW)
    amt_out = np.nan_to_num(A["amount_paid_base"].to_numpy(np.float64))
    amt_in = np.nan_to_num(A["amount_received_base"].to_numpy(np.float64))
    B = agg.set_b(src, dst, t, amt_out, amt_in, w, n_acc)
    timing["set_b"] = round(time.perf_counter() - tic, 1)
    progress(f"Set B: {B.shape[1]} columns ({timing['set_b']} s)")

    # One float32 matrix for all features; each part is released as soon as it is copied,
    # so the peak memory is the matrix plus one part rather than every copy at once.
    cols_by_set = {"A": list(A.columns), "B": list(B.columns),
                   "C": [f"{r}_{st}" for r in graph.ROLES for st in graph.STATS]}
    order = cols_by_set["A"] + cols_by_set["B"] + cols_by_set["C"]
    M = np.empty((n, len(order)), dtype=np.float32, order="F")
    j = 0
    for part in (A, B):
        for c in part.columns:
            M[:, j] = part[c].to_numpy()
            j += 1
    del A, B, part
    gc.collect()

    tic = time.perf_counter()
    sw = agg.window_minutes(P.SNAPSHOT_WINDOW)
    C, snaps, snap_of_row = graph.set_c(src, dst, t, origin, P.SNAPSHOT_FREQ, sw, progress)
    assert list(C.columns) == cols_by_set["C"]
    for c in C.columns:
        M[:, j] = C[c].to_numpy()
        j += 1
    del C
    gc.collect()
    snaps.to_csv(tables / f"{variant}_snapshots.csv", index=False)
    timing["set_c"] = round(time.perf_counter() - tic, 1)
    progress(f"Set C: {len(cols_by_set['C'])} columns ({timing['set_c']} s)")
    X = pd.DataFrame(M, columns=order, copy=False)          # a view, not a copy
    view = lambda k: X[cols_by_set[k]]

    tic = time.perf_counter()
    vrows = rng.choice(n, size=min(P.VERIFY_SAMPLE, n), replace=False)
    verification = {
        "set_b": agg.verify_b(src, dst, t, amt_out, amt_in, view("B"), w, vrows),
        "set_c": graph.verify_c(src, dst, t, view("C"), snap_of_row, sw,
                                vrows[: max(200, len(vrows) // 4)]),
        "snapshots_use_only_earlier_edges": bool(snaps["edges_strictly_before"].all()),
    }
    timing["verification"] = round(time.perf_counter() - tic, 1)
    if not (verification["set_b"]["passed"] and verification["set_c"]["passed"]
            and verification["snapshots_use_only_earlier_edges"]):
        raise RuntimeError(f"Causality verification failed: {verification}")
    progress(f"Causality verification passed ({timing['verification']} s)")

    fit_rows = np.flatnonzero(train & ~burn_in)
    sample = np.sort(rng.choice(fit_rows, size=min(P.SCREEN_SAMPLE, len(fit_rows)),
                                replace=False))
    kept, screen_report, rho = screening.screen(X.iloc[sample], order, P.SCREEN_THRESHOLD)
    rho.to_csv(tables / f"{variant}_feature_spearman.csv")
    sets = {"A": [c for c in cols_by_set["A"] if c in kept]}
    sets["B"] = sets["A"] + [c for c in cols_by_set["B"] if c in kept]
    sets["C"] = sets["B"] + [c for c in cols_by_set["C"] if c in kept]
    progress(f"Screening: {len(kept)} of {len(order)} columns kept; "
             f"sets A/B/C = {len(sets['A'])}/{len(sets['B'])}/{len(sets['C'])}")

    tic = time.perf_counter()
    writer = None
    for start in range(0, n, WRITE_CHUNK):
        sl = slice(start, min(start + WRITE_CHUNK, n))
        chunk = pd.DataFrame({"file_row": file_row[sl], "block": block[sl],
                              "is_laundering": label[sl], "burn_in": burn_in[sl],
                              "tail": tail[sl], "snapshot_minute": snap_of_row[sl],
                              **{c: M[sl, k] for k, c in enumerate(order)}})
        table = pa.Table.from_pandas(chunk, preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(features_dir / f"{variant}.parquet", table.schema)
        writer.write_table(table)
    writer.close()
    timing["write"] = round(time.perf_counter() - tic, 1)

    definition = {"variant": variant, "sets": sets,
                  "columns_before_screening": cols_by_set,
                  "parameters": {k: getattr(P, k) for k in dir(P) if k.isupper()}}
    (results_dir / f"feature_sets_{variant}.json").write_text(json.dumps(definition, indent=2))

    report = {
        "variant": variant,
        "n_rows": int(n), "n_burn_in_rows": int(burn_in.sum()),
        "n_training_rows_after_burn_in": int(len(fit_rows)),
        "tail": tail_info,
        "n_columns": {k: len(v) for k, v in cols_by_set.items()},
        "set_sizes_after_screening": {k: len(v) for k, v in sets.items()},
        "screening": screen_report,
        "fx": {"n_currencies": int(len(fx_diag)),
               "methods": fx_diag["method"].value_counts().to_dict(),
               "max_pair_iqr_log_rate": float(fx_pairs["iqr"].max()) if len(fx_pairs) else None},
        "snapshots": {"n": int(len(snaps)),
                      "rows_without_snapshot": int((snap_of_row < 0).sum())},
        "verification": verification,
        "timing_seconds": {**timing, "total": round(time.perf_counter() - t_start, 1)},
    }
    (results_dir / f"features_{variant}.json").write_text(json.dumps(report, indent=2, default=str))
    return report


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--variant", choices=config.VARIANTS, action="append")
    for v in ap.parse_args().variant or list(config.VARIANTS):
        print(json.dumps(build(v), indent=2, default=str))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile tests/test_features_bc.py
import json

import networkx as nx
import numpy as np
import pandas as pd
import pytest

from aml import aggregates as agg
from aml import fx, graph, screening


def toy_events(n=2500, n_acc=30, seed=0):
    rng = np.random.default_rng(seed)
    t = np.sort(rng.choice(np.arange(0, 3 * 1440, 9), size=n))     # many minute ties
    src, dst = rng.integers(0, n_acc, n), rng.integers(0, n_acc, n)
    amt = rng.lognormal(5, 2, n)
    return src.astype(np.int64), dst.astype(np.int64), t.astype(np.int64), amt, amt * 0.97


def naive_b(src, dst, t, ao, ai, w):
    """Independent reference: one boolean mask per transaction."""
    rows = []
    for r in range(len(t)):
        win = (t >= t[r] - w) & (t < t[r])
        o, i = win & (src == src[r]), win & (dst == dst[r])
        si, do = int((win & (dst == src[r])).sum()), int((win & (src == dst[r])).sum())
        lo, li = np.log1p(ao[o]), np.log1p(ai[i])

        def prev(a):
            ts = t[((src == a) | (dst == a)) & (t < t[r])]
            return (t[r] - ts.max()) / 60 if len(ts) else t[r] / 60
        rows.append({
            "src_out_cnt": o.sum(), "src_out_sum": ao[o].sum(),
            "src_out_mean_log": lo.mean() if len(lo) else 0,
            "src_out_std_log": lo.std(ddof=1) if len(lo) > 1 else 0,
            "src_out_n_cp": len(np.unique(dst[o])),
            "dst_in_cnt": i.sum(), "dst_in_sum": ai[i].sum(),
            "dst_in_mean_log": li.mean() if len(li) else 0,
            "dst_in_std_log": li.std(ddof=1) if len(li) > 1 else 0,
            "dst_in_n_cp": len(np.unique(src[i])),
            "src_in_share": (si + 1) / (si + o.sum() + 2),
            "dst_in_share": (i.sum() + 1) / (i.sum() + do + 2),
            "src_hours_since_prev": prev(src[r]), "dst_hours_since_prev": prev(dst[r])})
    return pd.DataFrame(rows)


def test_set_b_equals_naive_reference_with_ties():
    src, dst, t, ao, ai = toy_events()
    w = 1440
    got = agg.set_b(src, dst, t, ao, ai, w, n_accounts=30)
    ref = naive_b(src, dst, t, ao, ai, w)
    for c in ref.columns:
        np.testing.assert_allclose(got[c].to_numpy(), ref[c].to_numpy(), rtol=2e-5,
                                   atol=1e-5, err_msg=c)
    rep = agg.verify_b(src, dst, t, ao, ai, got, w, np.arange(0, len(t), 7))
    assert rep["passed"]


def test_window_boundaries():
    # account 0 pays 1 at t=0, 2 at t=60, 3 at t=100; queries at the same minutes
    src = np.array([0, 0, 0]); dst = np.array([1, 2, 1]); t = np.array([0, 60, 100])
    ao = np.array([1.0, 2.0, 3.0])
    got = agg.set_b(src, dst, t, ao, ao, w=60, n_accounts=3)
    assert got["src_out_cnt"].tolist() == [0, 1, 1]       # t=60 sees t=0 (= t - w); t=100 sees 60 only
    assert got["src_out_sum"].tolist() == [0, 1, 2]
    assert got["src_out_n_cp"].tolist() == [0, 1, 1]


def test_node_stats_match_networkx():
    rng = np.random.default_rng(3)
    u, v = rng.integers(0, 60, 400), rng.integers(0, 60, 400)
    keep = u != v
    u, v = u[keep] + 1000, v[keep] + 1000                   # non-contiguous codes
    nodes, st = graph.node_stats(u, v, chunk_budget=50)      # tiny budget: many chunks
    G = nx.DiGraph(list(zip(u.tolist(), v.tolist())))
    U = G.to_undirected()
    pr = nx.pagerank(G, alpha=0.85)
    cc = nx.clustering(U)
    for i, node in enumerate(nodes.tolist()):
        assert st["in_deg"][i] == G.in_degree(node)
        assert st["out_deg"][i] == G.out_degree(node)
        assert st["n_cp"][i] == U.degree(node)
        two = len(nx.single_source_shortest_path_length(U, node, cutoff=2)) - 1
        assert st["two_hop"][i] == two
        assert st["clustering"][i] == pytest.approx(cc[node])
        outs, ins = set(G.successors(node)), set(G.predecessors(node))
        assert st["reciprocity"][i] == pytest.approx(len(outs & ins) / len(outs | ins))
        assert st["pagerank"][i] == pytest.approx(pr[node] * len(nodes))
        assert st["component_size"][i] == len(nx.node_connected_component(U, node))


def test_set_c_uses_only_earlier_snapshots():
    src, dst, t, _, _ = toy_events(n=1500)
    origin = pd.Timestamp("2022-09-01 06:30")                # not a midnight
    C, snaps, snap_of_row = graph.set_c(src, dst, t, origin, "1D", 1440, progress=lambda m: None)
    assert (snap_of_row[snap_of_row >= 0] <= t[snap_of_row >= 0]).all()
    assert snaps["edges_strictly_before"].all()
    first = int(snaps["snapshot_minute"].min())
    assert first == 17 * 60 + 30                             # next midnight after 06:30
    before = t < first
    assert (C.loc[before, "src_out_deg"] == 0).all() and (C.loc[before, "src_component_size"] == 1).all()
    rep = graph.verify_c(src, dst, t, C, snap_of_row, 1440, np.arange(len(t)))
    assert rep["passed"] and rep["n_rows_checked"] > 0


def test_fx_rates_through_a_chain():
    rows = []
    for _ in range(30):   # EUR<->USD at 1.1 USD per EUR, GBP->EUR at 1.2 EUR per GBP
        rows += [("Euro", "US Dollar", 100.0, 110.0), ("US Dollar", "Euro", 110.0, 100.0),
                 ("UK Pound", "Euro", 10.0, 12.0), ("US Dollar", "US Dollar", 5.0, 5.0)]
    df = pd.DataFrame(rows, columns=["Payment Currency", "Receiving Currency",
                                     "Amount Paid", "Amount Received"])
    for c in ["Payment Currency", "Receiving Currency"]:
        df[c] = df[c].astype("category")
    rates, diag, pairs = fx.implied_rates(df, "US Dollar")
    assert rates["US Dollar"] == pytest.approx(1.0)
    assert rates["Euro"] == pytest.approx(1.1)
    assert rates["UK Pound"] == pytest.approx(1.32)
    assert (diag["method"] == "implied rate").all()
    usd = fx.to_base(df["Amount Paid"], df["Payment Currency"], rates)
    assert usd[2] == pytest.approx(13.2)


def test_screening_drops_later_duplicates_only():
    rng = np.random.default_rng(0)
    a = rng.normal(size=5000)
    X = pd.DataFrame({"A_x": a, "A_const": 1.0, "A_y": rng.normal(size=5000),
                      "B_dup": 2 * a + 1e-6 * rng.normal(size=5000), "C_z": rng.normal(size=5000)})
    kept, rep, _ = screening.screen(X, list(X.columns), 0.95)
    assert kept == ["A_x", "A_y", "C_z"]
    assert rep["constant"] == ["A_const"] and rep["redundant"]["B_dup"]["duplicates"] == "A_x"


def test_build_end_to_end(tmp_path):
    from aml import build_features, prepare
    from tests.synthetic import write_synthetic
    write_synthetic(tmp_path / "raw", n=6000, seed=1)
    prepare.prepare("HI-Small", tmp_path / "raw", tmp_path / "interim", tmp_path / "results")
    rep = build_features.build("HI-Small", tmp_path / "interim", tmp_path / "features",
                               tmp_path / "results", progress=lambda m: None)
    assert rep["verification"]["set_b"]["passed"] and rep["verification"]["set_c"]["passed"]
    sets = json.loads((tmp_path / "results" / "feature_sets_HI-Small.json").read_text())["sets"]
    assert set(sets["A"]) <= set(sets["B"]) <= set(sets["C"])
    X = pd.read_parquet(tmp_path / "features" / "HI-Small.parquet")
    assert len(X) == 6000 and "typology" not in X.columns
    assert not X[sets["C"]].isna().any().any()


def test_tail_flag_and_weekend():
    from aml.build_features import tail_flag
    days = pd.to_datetime(["2022-09-01"] * 1000 + ["2022-09-02"] * 900 + ["2022-09-03"] * 950
                          + ["2022-09-04"] * 12 + ["2022-09-05"] * 3)
    tail, last = tail_flag(pd.Series(days), 0.10)
    assert str(last.date()) == "2022-09-03" and tail.sum() == 15 and tail[-15:].all()
    from aml.features import base_attributes
    ts = pd.to_datetime(["2022-09-02 10:00", "2022-09-03 10:00", "2022-09-04 10:00"])
    df = pd.DataFrame({"Timestamp": ts, "Amount Paid": 1.0, "Amount Received": 1.0,
                       "Payment Currency": pd.Categorical(["US Dollar"] * 3),
                       "Receiving Currency": pd.Categorical(["US Dollar"] * 3),
                       "Payment Format": pd.Categorical(["ACH"] * 3),
                       "From Bank": pd.Categorical(["1"] * 3), "To Bank": pd.Categorical(["2"] * 3)})
    from aml.build_features import set_a
    A = set_a(df, pd.Series({"US Dollar": 1.0}), np.ones(3, bool), "weekend")
    assert A["weekend"].tolist() == [0, 1, 1] and "day_of_week" not in A


## 1 Design decisions

The cell below fixes every parameter of feature construction. It revises Section 4.6 in four
places, each forced by the length of the observed period (about ten days) rather than chosen
after seeing any result, and each to be reported as a revision of the specification:

| Specification (Section 4.6) | Here | Reason |
| --- | --- | --- |
| Set B window of 30 days | 24 hours | A 30-day window never rolls in a 10-day period: every aggregate would be cumulative and would grow with calendar time, so values in the test block would exceed those seen in training. |
| Weekly graph snapshots from all earlier transactions | daily snapshots, each from the preceding 24 hours | With one snapshot on day 7, almost the whole training block would have no graph statistics; cumulative graphs drift for the same reason as cumulative aggregates. |
| — | first 24 hours excluded from model fitting | No history exists yet for Sets B and C; the same rows are excluded for all three sets. |
| Raw amounts per currency | amounts in US dollars, at rates implied by cross-currency transactions of the training block | Amounts in different currencies are not comparable (Notebook 01, §4.6). |

Two further decisions follow from Notebook 01. The day of the week is kept only as a weekend
indicator: single weekdays occur once or twice, but the weekend recurs in the training and test
blocks with a laundering rate about twice the weekday rate (`DAY_OF_WEEK`). And regular activity
ends on 10 September; the sparse remainder, mostly laundering, is flagged as `tail` and excluded
from the primary evaluation in Notebook 03 (`TAIL_SHARE`). Means and standard deviations in
Set B are taken of log(1 + amount), which is numerically stable for amounts spanning many
orders of magnitude.

Edit values only before Notebook 03 is run; afterwards any change is a deviation for Appendix A.

In [ ]:
%%writefile aml/feature_params.py
"""Feature construction parameters (Section 4.6, revised after the EDA of Notebook 01).

These values are fixed before any model is trained and are not tuned. Changing one
after Notebook 03 has produced results is a deviation that must be reported in Appendix A.
"""
# Set B: account aggregates over a look-back window that ends strictly before the
# transaction. The observed period is about ten days, so a 30-day window would never
# roll (every aggregate would be cumulative) and is replaced by 24 hours.
WINDOW = "24h"

# Set C: one graph snapshot per day at midnight, each built from the transactions of the
# preceding SNAPSHOT_WINDOW only. Weekly snapshots would leave almost the whole training
# block without a snapshot; cumulative graphs would grow with calendar time.
SNAPSHOT_FREQ = "1D"
SNAPSHOT_WINDOW = "24h"

# The first BURN_IN of the observed period has no complete history for Sets B and C.
# Its rows are flagged and excluded from model fitting for every feature set alike.
BURN_IN = "24h"

# Set A: amounts are converted into one currency with the exchange rates implied by
# cross-currency transactions in the training block, so that they are comparable.
BASE_CURRENCY = "US Dollar"

# Set A: day of the week. Each weekday occurs only once or twice in ten days of regular
# activity, so the full day of the week nearly identifies the calendar date. The weekend
# pattern recurs, however (training block: 3-4 September, a laundering rate about twice
# the weekday rate; test block: 10 September), so it is kept as a weekend indicator.
# Options: "weekend" (default), "drop", "full".
DAY_OF_WEEK = "weekend"

# End of the observation period. After the last calendar day whose transaction count is at
# least TAIL_SHARE of the median daily count, the generator's licit activity stops while
# laundering attempts run to completion (HI-Small: about 1,100 transactions over eight
# days, most of them laundering). These rows are flagged `tail`; Notebook 03 excludes them
# from the primary evaluation and reports them separately. The rule uses counts only,
# never labels.
TAIL_SHARE = 0.10

# Screening (training block, after burn-in): constant features are dropped, and of each
# pair with |Spearman rho| above the threshold the later feature in A -> B -> C order.
SCREEN_THRESHOLD = 0.95
SCREEN_SAMPLE = 1_000_000      # training rows sampled for the correlation matrix

# Causality verification: rows whose features are recomputed by brute force.
VERIFY_SAMPLE = 2_000
SEED = 0


In [ ]:
# Use the modules just written: discard any copy imported earlier in this session
import importlib
for name in [n for n in list(sys.modules) if n.split(".")[0] in ("aml", "tests")]:
    del sys.modules[name]
importlib.invalidate_caches()

In [ ]:
# Run the test suite (Notebook 01 and Notebook 02 tests; synthetic data only)
res = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"],
                     capture_output=True, text=True)
print(res.stdout[-2500:], res.stderr[-2000:])
assert res.returncode == 0, "tests failed"

## 2 Building the feature matrices

For each variant, independently: Set A is encoded, Sets B and C are computed, a random sample
of rows is recomputed by brute force to verify causality (the build stops if any value
differs), the screening rule is applied on the training block, and the full matrix is written
to `data/features/<variant>.parquet` together with the set definitions in
`results/feature_sets_<variant>.json`.

In [ ]:
# Features: build, verify, screen and store both variants
from aml import build_features, config
from aml import feature_params as P

if DRY_RUN:   # synthetic stand-in for Notebook 01's output
    from aml import prepare
    from tests.synthetic import write_synthetic
    for i, v in enumerate(VARIANTS):
        if not (PROJECT / "data" / "interim" / f"{v}.parquet").exists():
            write_synthetic(PROJECT / "data" / "raw", variant=v, n=200_000, seed=i)
            prepare.prepare(v)

missing = [v for v in VARIANTS if not (PROJECT / "data" / "interim" / f"{v}.parquet").exists()]
assert not missing, f"prepared data missing for {missing}: run Notebook 01 first"

reports = {}
for v in VARIANTS:
    print(f"=== {v}")
    reports[v] = build_features.build(v)
    gc.collect()

display(pd.DataFrame({v: {"rows": r["n_rows"], "burn-in rows": r["n_burn_in_rows"],
                          "tail rows (after " + r["tail"]["last_regular_day"] + ")": r["tail"]["n_rows"],
                          "training rows after burn-in": r["n_training_rows_after_burn_in"],
                          **{f"columns {k}": n for k, n in r["n_columns"].items()},
                          **{f"set {k} after screening": n
                             for k, n in r["set_sizes_after_screening"].items()},
                          "minutes": round(r["timing_seconds"]["total"] / 60, 1)}
                      for v, r in reports.items()}).astype(object))

In [ ]:
# Features: load the explored variant (training block after burn-in) for Sections 3 to 7
definition = json.loads((PROJECT / "results" / f"feature_sets_{EDA_VARIANT}.json").read_text())
fit = pd.read_parquet(PROJECT / "data" / "features" / f"{EDA_VARIANT}.parquet",
                      filters=[("block", "==", "train"), ("burn_in", "==", False)])
y = fit["is_laundering"].to_numpy()
rng = np.random.default_rng(SEED)
neg = np.flatnonzero(y == 0)
plot_idx = np.concatenate([np.flatnonzero(y == 1),
                           rng.choice(neg, size=min(100_000, len(neg)), replace=False)])
FIG, TAB = PROJECT / "results" / "figures", PROJECT / "results" / "tables"
def save(fig, name):
    fig.savefig(FIG / f"{EDA_VARIANT}_{name}.png", dpi=200, bbox_inches="tight")
print(f"training rows after burn-in: {len(fit):,} ({y.sum():,} laundering)")

## 3 Set A: transaction attributes

In [ ]:
# Set A: exchange rates implied by cross-currency transactions in the training block
fx_rates = pd.read_csv(TAB / f"{EDA_VARIANT}_fx_rates.csv", index_col=0)
fx_pairs = pd.read_csv(TAB / f"{EDA_VARIANT}_fx_pairs.csv")
display(fx_rates)
print("Dispersion of the implied log rate within each currency pair "
      "(interquartile range; 0 means one fixed rate):")
display(fx_pairs.sort_values("n", ascending=False).head(15))

If every pair shows an interquartile range near zero, the generator uses fixed exchange rates
and the conversion is exact. A currency marked *median scaling* had no chain of
cross-currency transactions to the US dollar; report it in Appendix A.

In [ ]:
# Set A: numeric attributes by class (training block after burn-in)
num_a = [c for c in definition["columns_before_screening"]["A"]
         if not c.startswith(("cur_pay=", "cur_rec=", "fmt="))]
by_class = fit[num_a].groupby(y).median().T.rename(columns={0: "licit", 1: "laundering"})
display(by_class.round(3))
by_class.to_csv(TAB / f"{EDA_VARIANT}_setA_median_by_class.csv")

## 4 Set B: account aggregates over the preceding 24 hours

Fourteen aggregates: for the originating account, its outgoing transactions; for the
beneficiary account, its incoming transactions (count, summed amount, mean and standard
deviation of log amount, distinct counterparties); the share of incoming activity of each
account; and the hours since each account's previous transaction. Every value uses only
transactions in the 24 hours strictly before the transaction it describes.

In [ ]:
# Set B: median by class and standardised mean difference (training block after burn-in)
cols = definition["columns_before_screening"]["B"]
g = fit[cols].groupby(y)
summary = pd.DataFrame({"median licit": g.median().loc[0], "median laundering": g.median().loc[1],
                        "mean licit": g.mean().loc[0], "mean laundering": g.mean().loc[1]})
summary["std. mean difference"] = ((summary["mean laundering"] - summary["mean licit"])
                                   / fit[cols].std().replace(0, np.nan))
display(summary.round(3))
summary.to_csv(TAB / f"{EDA_VARIANT}_setB_by_class.csv")

In [ ]:
# Set B: distributions by class (all laundering rows, random sample of licit rows)
cols = definition["columns_before_screening"]["B"]
S = fit.iloc[plot_idx][cols].copy()
S["class"] = np.where(y[plot_idx] == 1, "laundering", "licit")
n_col = 4
fig, axes = plt.subplots(int(np.ceil(len(cols) / n_col)), n_col,
                         figsize=(16, 3.2 * int(np.ceil(len(cols) / n_col))))
for ax, c in zip(axes.ravel(), cols):
    vals = S[c].astype(float)
    shift = 1.0 - min(vals.min(), 0.0)      # log scale needs positive values
    sns.boxplot(x=S["class"], y=vals + shift, ax=ax, showfliers=False,
                order=["licit", "laundering"])
    ax.set_yscale("log")
    ax.set_title(c, fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("")
for ax in axes.ravel()[len(cols):]:
    ax.set_visible(False)
fig.suptitle("Set B: distributions by class (log scale, value + shift)", y=1.0)
fig.tight_layout()
save(fig, "setB_by_class")
plt.show()

## 5 Set C: account-graph statistics from daily snapshots

In [ ]:
# Set C: the daily snapshots (edges are transactions of the preceding 24 hours)
snaps = pd.read_csv(TAB / f"{EDA_VARIANT}_snapshots.csv")
origin = pd.Timestamp(json.loads((PROJECT / "results" / f"prepare_{EDA_VARIANT}.json")
                                 .read_text())["integrity"]["first_timestamp"])
snaps.insert(0, "snapshot_time", origin + pd.to_timedelta(snaps["snapshot_minute"], unit="min"))
display(snaps.drop(columns="snapshot_minute"))

In [ ]:
# Set C: median by class and standardised mean difference (training block after burn-in)
cols = definition["columns_before_screening"]["C"]
g = fit[cols].groupby(y)
summary = pd.DataFrame({"median licit": g.median().loc[0], "median laundering": g.median().loc[1],
                        "mean licit": g.mean().loc[0], "mean laundering": g.mean().loc[1]})
summary["std. mean difference"] = ((summary["mean laundering"] - summary["mean licit"])
                                   / fit[cols].std().replace(0, np.nan))
display(summary.round(3))
summary.to_csv(TAB / f"{EDA_VARIANT}_setC_by_class.csv")

In [ ]:
# Set C: distributions by class (all laundering rows, random sample of licit rows)
cols = definition["columns_before_screening"]["C"]
S = fit.iloc[plot_idx][cols].copy()
S["class"] = np.where(y[plot_idx] == 1, "laundering", "licit")
n_col = 4
fig, axes = plt.subplots(int(np.ceil(len(cols) / n_col)), n_col,
                         figsize=(16, 3.2 * int(np.ceil(len(cols) / n_col))))
for ax, c in zip(axes.ravel(), cols):
    vals = S[c].astype(float)
    shift = 1.0 - min(vals.min(), 0.0)      # log scale needs positive values
    sns.boxplot(x=S["class"], y=vals + shift, ax=ax, showfliers=False,
                order=["licit", "laundering"])
    ax.set_yscale("log")
    ax.set_title(c, fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("")
for ax in axes.ravel()[len(cols):]:
    ax.set_visible(False)
fig.suptitle("Set C: distributions by class (log scale, value + shift)", y=1.0)
fig.tight_layout()
save(fig, "setC_by_class")
plt.show()

## 6 Causality verification

This replaces the assertion of Listing 4.4. For a random sample of rows, every Set B feature
is recomputed by filtering the account's raw transactions directly, and the degree and
counterparty statistics of Set C are recomputed from the raw transactions of the snapshot
window; the build stops if any value differs. Every snapshot is also checked to contain only
transactions strictly earlier than the snapshot time.

In [ ]:
# Verification: results for both variants
for v, r in reports.items():
    ver = r["verification"]
    print(f"{v}: Set B {'passed' if ver['set_b']['passed'] else 'FAILED'} on {ver['set_b']['n_rows']} rows "
          f"(largest relative error {max(ver['set_b']['max_relative_error'].values()):.1e}); "
          f"Set C {'passed' if ver['set_c']['passed'] else 'FAILED'} on {ver['set_c']['n_rows_checked']} rows; "
          f"snapshots strictly earlier: {ver['snapshots_use_only_earlier_edges']}")

## 7 Feature selection

The course examples select features with RFE or `SelectKBest` fitted per model. Here the three
nested sets are the experimental factor of RQ2, so selection may not differ between model
families or break the nesting. Three parts, fixed in advance (Section 4.6):

1. **Screening** (this section): on the training block after burn-in, constant features are
   dropped, and of each pair with |Spearman ρ| > 0.95 the later feature in A → B → C order.
   The same result applies to every family.
2. **Embedded selection** (Notebook 03): L1 and elastic-net penalties in the logistic
   regression search.
3. **Diagnostics, not selection** (this section and Notebook 04): mutual information and
   post hoc importances are reported but never used to remove features.

### 7.1 Screening

In [ ]:
# Screening: what was removed, and why
scr = reports[EDA_VARIANT]["screening"]
print(f"rows used: {scr['n_rows']:,}   candidates: {scr['n_candidates']}   kept: {scr['n_kept']}")
print("constant:", scr["constant"] or "none")
red = pd.DataFrame(scr["redundant"]).T
display(red if len(red) else "no redundant pairs above the threshold")

In [ ]:
# Screening: Spearman correlation matrix of all candidate features (training block)
rho = pd.read_csv(TAB / f"{EDA_VARIANT}_feature_spearman.csv", index_col=0)
fig, ax = plt.subplots(figsize=(18, 15))
sns.heatmap(rho, cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True, ax=ax,
            xticklabels=True, yticklabels=True, cbar_kws={"shrink": 0.6})
ax.tick_params(labelsize=6)
ax.set_title("Spearman correlation of candidate features, training block after burn-in")
save(fig, "feature_spearman")
plt.show()

### 7.2 Mutual information (diagnostic)

In [ ]:
# Diagnostic: mutual information with the label, as SelectKBest(mutual_info_classif) in the course
from sklearn.feature_selection import SelectKBest, mutual_info_classif

kept = definition["sets"]["C"]
Xs = fit.iloc[plot_idx][kept].to_numpy(dtype=float)
ys = y[plot_idx]
discrete = np.array([fit[c].nunique() <= 2 for c in kept])
selector = SelectKBest(lambda X, Y: mutual_info_classif(X, Y, discrete_features=discrete,
                                                        random_state=SEED), k="all").fit(Xs, ys)
mi = pd.Series(selector.scores_, index=kept).sort_values(ascending=False)
mi.to_csv(TAB / f"{EDA_VARIANT}_mutual_information.csv")
set_of = {c: s for s in "CBA" for c in definition["sets"][s]}
fig, ax = plt.subplots(figsize=(10, 9))
top = mi.head(30)[::-1]
ax.barh(top.index, top.values, color=[{"A": "tab:blue", "B": "tab:orange", "C": "tab:green"}[set_of[c]]
                                      for c in top.index])
ax.set_xlabel("mutual information with the label (nats)")
ax.set_title("Top 30 features by mutual information (blue A, orange B, green C)")
ax.tick_params(axis="y", labelsize=8)
save(fig, "mutual_information")
plt.show()

The sample contains every laundering row of the training block and 100,000 licit rows, so
the values are inflated relative to the true base rate; only the ranking is informative. Mutual
information is univariate: a feature that is uninformative alone can still matter in
combination, which is why it is reported and not used to drop features.

### 7.3 Final feature sets

In [ ]:
# Feature sets after screening (identical for every model family)
sets = definition["sets"]
for name in "ABC":
    new = [c for c in sets[name] if name == "A" or c not in sets[chr(ord(name) - 1)]]
    print(f"Set {name}: {len(sets[name])} columns ({len(new)} added)")
    print("   ", ", ".join(new))

## 8 Findings to record

1. Section 4.6: replace the 30-day window, the weekly cumulative snapshots and the raw amounts
   by the parameters of Section 1, with the reasons given there, and add the burn-in rule and
   the screening rule. These revisions precede any model result.
2. Exchange rates (Section 3): fixed or varying? Any currency converted by median scaling?
3. Which Set B and Set C features differ most between the classes (standardised mean
   difference, Sections 4 and 5)?
4. Screening (7.1): which features were removed, and as duplicates of which?
5. Mutual information (7.2): which set contributes the highest-ranked features? (Diagnostic
   only; RQ2 is answered in Notebook 03.)
6. Appendix A: causality verification results (Section 6) and run times per variant.
7. Section 4.5: the observation period ends on the last day of regular activity; the number of
   tail rows and the laundering transactions among them (Section 2) are reported, and the tail
   is evaluated separately in Notebook 03.

## 9 Output to send back

In [ ]:
# Output: build reports and feature-set definitions for both variants
for v in VARIANTS:
    for name in (f"features_{v}.json", f"feature_sets_{v}.json"):
        print(f"----- {name}")
        print((PROJECT / "results" / name).read_text())